In [1]:
import torch
from mario_gpt import MarioDataset, MarioLM, TrainingConfig, MarioGPTTrainer
from mario_gpt.utils import view_level, convert_level_to_png, join_list_of_list, characterize

In [2]:
mario_lm = MarioLM()
dataset = MarioDataset(tokenizer=mario_lm.tokenizer, folder_path='/mario-gpt/txt')

Using shyamsn97/Mario-GPT2-700-context-length lm


/opt/conda/lib/python3.10/site-packages/transformers-4.46.3-py3.10.egg/transformers/models/auto/modeling_auto.py:1833: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


Using shyamsn97/Mario-GPT2-700-context-length tokenizer


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Token indices sequence length is longer than the specified maximum sequence length for this model (3360 > 1024). Running this sequence through the model will result in indexing errors


In [3]:
view_level(dataset.data[1]["input_ids"][:700], mario_lm.tokenizer)

['--------------------------------------------------',
 '--------------------------------------------------',
 '--------------------------------------------------',
 '--------------------------------------------------',
 '--------------------------------------------------',
 '----------------------------------------C---------',
 '--------------------------------C-----------------',
 '-----------------------------oo-----------------oo',
 '----------------------------o------------------o--',
 '------------!---?!------oo-------------SCS--------',
 '-----------------------o-----XXXXXX---------------',
 '-----------------------------XXXXXX---------------',
 '------------------g-----XXXXXXXXXXX------------XXX',
 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX']

In [4]:
!pip -q install --upgrade huggingface_hub 
!apt -q install git -y
!pip -q install groq
!pip -q install python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading package lists...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Building dependency tree...
Reading state information...
The following additional packages will be installed:
  git-man less libcurl3-gnutls liberror-perl libnghttp2-14 libpsl5 librtmp1
  libssl1.0.0 openssh-client publicsuffix xauth
Suggested packages:
  gettext-base git-daemon-run | git-daemon-sysvinit git-doc git-el git-email
  git-gui gitk gitweb git-cvs git-mediawiki git-svn keychain libpam-ssh
  monkeysphere ssh-askpass
The following NEW packages will be installed:
  git git-man less libcurl3-gnutls liberror-perl libnghttp2-14 libpsl5
  librtmp1 libssl1.0.0 openssh-client publicsuffix xauth
0 upgraded, 12 newly installed, 0 to remove and 58 not upgraded.
Need to get 7142 kB of archives.
After this operation, 43.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu bionic/main amd64 less amd64 487-0.1 [112 kB]
Get:2 http://archive.ubuntu.com/ubuntu bionic/main amd64 libpsl5 amd64 0.19.1-5build1 [41.8 kB]
Get:3 http://archive.ubuntu.com/ubuntu bionic-u

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
from mario_gpt.prompter import Prompter
from mario_gpt.prompt_adapter import PromptAdapter
from transformers import pipeline
from groq import Groq
from tqdm import tqdm
from transformers import AutoTokenizer
# from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv()

# Pre-initialize the LLM model
# login(token=os.getenv("HUGGINGFACE_TOKEN"))
# llm_model = pipeline("text-generation", model="meta-llama/Llama-3.1-8B", device=0)
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
tokenizer_path = "shyamsn97/Mario-GPT2-700-context-length"
level_tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
prompter = Prompter(level_tokenizer=level_tokenizer)
adapter = PromptAdapter(llm=client, prompter=prompter)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [10]:
import json
import torch
from typing import Tuple

def get_item_from_dataset(dataset, indice: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Retrieves an item (input_ids and attention_mask) from the dataset at a given index.
    """
    input_ids, attention_mask = dataset[indice]
    return torch.stack([input_ids]), torch.stack([attention_mask])

def save_dataset_to_json(dataset, output_file: str):
    """
    Saves the first two indexed items from the dataset into a JSON file.
    """
    data_list = []
    for i in range(2):  # Get first two items
        input_ids, attention_masks = get_item_from_dataset(dataset, i)
        
        for level in input_ids:
            prompt_base, encoder_hidden_state, _, str_level = prompter(level=level)  
            adapted_prompt = adapter.adapt_prompt(prompt_base, True)
            # print(adapted_prompt)

            data_list.append({
                "input_ids": level.tolist(),  
                "prompt": prompt_base,
                "adapted_prompt": adapted_prompt,
                "encoder_hidden_state": encoder_hidden_state.tolist(),
                "str_level": str_level
            })

    # Save to JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data_list, f, indent=4)


In [11]:
save_dataset_to_json(dataset, "mario_dataset.json")

('I want a level with three pipes, five enemies, 106 blocks, no Koopas, no Goombas, one power-up, nine coins, and a low elevation.', 'I want a level with 3 pipes, 5 enemys, 106 blocks, 0 koopas, 0 goombas, 1 powerups, 9 coins, Unknown, low elevation')
('Create a level that has 3 pipes, 5 enemies, 106 blocks, 0 Koopas, 0 Goombas, 1 power-up, 9 coins, an unknown element, and a low elevation.', 'Create a level that has 3 pipes, 5 enemys, 106 blocks, 0 koopas, 0 goombas, 1 powerups, 9 coins, Unknown, low elevation')
